# L1B Radiometer Algorithm

This notebook walks through the full L1B processing algorithm for the LIBERA radiometer. From L1A radiometer data, housekeeping data, calibration coefficients generated by the instrument engineering team, and geolocation data, the l1b algorithm produces a data product with radiance values and other information as defined in the product definition. Starting from L1A radiometer and housekeeping NetCDF files, the algorithm applies gain calibration, downsamples the signal from 200 Hz to 100 Hz, and converts digital numbers (DN) into physical radiance values (W m⁻² sr⁻¹). For visualizations and additional details on this algorithm, see [Filtered Radiance Algorithm Diagram](https://lasp.colorado.edu/galaxy/x/YQZgD).

**Inputs:**
- L1A radiometer data (4-channel DN time series at 200 Hz)
- L1A housekeeping data (FPE telescope temperature at ~1 Hz)
- Ground calibration coefficients (loaded internally by `libera_rad`)

Note: in production, goelocation data is also an input, but is not analyzed in this notebook since it is not required to calculate radiance.

**Output:** Calibrated radiance for four channels — Shortwave (SW), Total, Longwave (LW), and Split Shortwave (SSW) — at 100 Hz.

**Processing steps:**
1. Load input files
2. Extract radiometer DN and temperature data
3. Interpolate housekeeping temperature to the radiometer time grid
4. Apply gain calibration in the frequency domain
5. Downsample from 200 Hz to 100 Hz
6. Convert calibrated DN to radiance
7. Validate output against a reference L1B file

## Setup

Import dependencies and configure the plot style. From `libera_rad`, the 'gain_calibration' module provides the instrument-specific calibration routines and the `radiance` module provides the DN to radiance conversion routines.

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from libera_rad.radiometer import gain_calibration, radiance
from scipy.fft import rfftfreq

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

---
## Step 1 — Load Input Files

The algorithm requires two L1A NetCDF files produced by the upstream decode pipeline. Both files share the same observation window and are identified by a common timestamp range and revision number in their filenames.

### A. L1A Radiometer Data

Contains the raw 4-channel DN samples at 200 Hz (`ICIE__RAD_SAMPLE_0` through `_3`) and the corresponding FPE timestamps (`RAD_SAMPLE_FPE_TIME`).

In [ ]:
print("Loading radiometer L1A data...")
radiometer_file = Path.cwd().parent / "tests" / "test_data" / "l1b_integration_data" / "LIBERA_L1A_RAD-SAMPLE-DECODED_V5-4-2_20251120T175950_20251120T190549_R26016183821.nc"
radiometer_data = xr.open_dataset(radiometer_file)
print(f"Dataset loaded successfully")
print(f"\nDataset dimensions: {dict(radiometer_data.sizes)}")

### B. L1A Housekeeping Data

Contains slow-cadence (~1 Hz) instrument health and environment telemetry. The field used here is `ICIE__FPE_TSCOPE_TEMP`, the focal-plane electronics telescope temperature in DN units. This temperature is required to convert DN to radiance because the detector sensitivity is temperature-dependent.

In [ ]:
print("Loading housekeeping L1A data...")
housekeeping_file = Path.cwd().parent / "tests" / "test_data" / "l1b_integration_data" / "LIBERA_L1A_NOM-HK-DECODED_V5-4-2_20251120T175950_20251120T190549_R26016183821.nc"
housekeeping_data = xr.open_dataset(housekeeping_file)
print(f"Dataset loaded successfully")
print(f"\nDataset dimensions: {dict(housekeeping_data.sizes)}")

---
## Step 2 — Extract and Inspect Raw Data

### A. Radiometer DN Channels

Pull each channel array out of the xarray dataset into numpy arrays and build a time-indexed pandas DataFrame. The four channels measure different portions of the electromagnetic spectrum:

| Channel | Variable | Band |
|---------|----------|------|
| CH0 | `ICIE__RAD_SAMPLE_0` | Shortwave (SW) |
| CH1 | `ICIE__RAD_SAMPLE_1` | Total |
| CH2 | `ICIE__RAD_SAMPLE_2` | Longwave (LW) |
| CH3 | `ICIE__RAD_SAMPLE_3` | Split Shortwave (SSW) |

More information on each channel can be found in `libera_rad.data.l1b_ground_calibration.json`.

All channels are sampled at **200 Hz** (5 ms intervals).

In [ ]:
# Extract radiometer data for all 4 channels (in DN units)
rad_ch0 = radiometer_data['ICIE__RAD_SAMPLE_0'].values  # Channel 0
rad_ch1 = radiometer_data['ICIE__RAD_SAMPLE_1'].values  # Channel 1
rad_ch2 = radiometer_data['ICIE__RAD_SAMPLE_2'].values  # Channel 2
rad_ch3 = radiometer_data['ICIE__RAD_SAMPLE_3'].values  # Channel 3

# Extract timestamps (200 Hz sample rate)
radiometer_timestamps = pd.to_datetime(radiometer_data['RAD_SAMPLE_FPE_TIME'].values)

# Extract observation ID
obsid = radiometer_data['ICIE__RAD_OBSID_RAD'].values

print(f"Data loaded:")
print(f"  Number of samples: {len(radiometer_timestamps):,}")
print(f"  Sample rate: 200 Hz (5 ms intervals)")
print(f"  Duration: {(radiometer_timestamps[-1] - radiometer_timestamps[0]).total_seconds():.1f} seconds")
print(f"  Start time: {radiometer_timestamps[0]}")
print(f"  End time: {radiometer_timestamps[-1]}")
print(f"\nChannel statistics (DN units):")
print(f"  CH0: mean={np.mean(rad_ch0):.2f}, std={np.std(rad_ch0):.2f}, range=[{np.min(rad_ch0):.2f}, {np.max(rad_ch0):.2f}]")
print(f"  CH1: mean={np.mean(rad_ch1):.2f}, std={np.std(rad_ch1):.2f}, range=[{np.min(rad_ch1):.2f}, {np.max(rad_ch1):.2f}]")
print(f"  CH2: mean={np.mean(rad_ch2):.2f}, std={np.std(rad_ch2):.2f}, range=[{np.min(rad_ch2):.2f}, {np.max(rad_ch2):.2f}]")
print(f"  CH3: mean={np.mean(rad_ch3):.2f}, std={np.std(rad_ch3):.2f}, range=[{np.min(rad_ch3):.2f}, {np.max(rad_ch3):.2f}]")

# Create a DataFrame with DN values and timestamps for easier manipulation
radiometer_dns = pd.DataFrame({
    'time': radiometer_timestamps,
    'dn_ch0': rad_ch0,
    'dn_ch1': rad_ch1,
    'dn_ch2': rad_ch2,
    'dn_ch3': rad_ch3,
})

radiometer_dns.set_index('time', inplace=True)

The plots below show the raw DN time series for each channel. Each channel is displayed individually (stacked subplots) and then overlaid on a single axes for direct amplitude comparison. At this stage no calibration has been applied — the values are raw integer counts from the ADC.

In [ ]:
# Select a subset of data for plotting (first 60 seconds = 12,000 samples at 200 Hz)
n_samples_plot = len(radiometer_dns) # Change this to shorten data shown in plot
df_plot = radiometer_dns.iloc[:n_samples_plot]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'Radiometer DN Time Series (200 Hz)',
             fontsize=14, fontweight='bold')

channels = ['ch0', 'ch1', 'ch2', 'ch3']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
channel_names = ['Shortwave', 'Total', 'Longwave', 'Split Shortwave']
for i, (ch, color) in enumerate(zip(channels, colors)):
    ax = axes[i]
    ax.plot(df_plot.index, df_plot[f'dn_{ch}'], color=color, linewidth=0.5, alpha=0.8)
    ax.set_ylabel(f'{channel_names[i]}\nDN', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([df_plot[f'dn_{ch}'].min() - 10, df_plot[f'dn_{ch}'].max() + 10])

    # Add statistics text
    mean_val = df_plot[f'dn_{ch}'].mean()
    std_val = df_plot[f'dn_{ch}'].std()
    min_val = df_plot[f'dn_{ch}'].min()
    max_val = df_plot[f'dn_{ch}'].max()
    ax.text(0.02, 0.95, f'μ={mean_val:.1f}, σ={std_val:.2f}\nmin={min_val:.1f}, max={max_val:.1f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[-1].set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
axes[-1].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {n_samples_plot:,} samples ({n_samples_plot/200:.1f} seconds)")


# Plot all channels on the same axes for comparison
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle(f'All Radiometer Channels DN (200 Hz)',
             fontsize=14, fontweight='bold')

for i, (ch, color) in enumerate(zip(channels, colors)):
    ax.plot(df_plot.index, df_plot[f'dn_{ch}'], color=color, linewidth=0.8,
            alpha=0.7, label=f'{channel_names[i]}')

ax.set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
ax.set_ylabel('DN (Digital Number)', fontsize=11, fontweight='bold')
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### B. Housekeeping Temperature

Extract the FPE telescope temperature from the housekeeping dataset. The temperature is sampled at a much lower rate (~1 Hz) than the radiometer (200 Hz), so it must be interpolated onto the radiometer time grid before it can be used in the radiance conversion (Step 5).


In [ ]:
# Extract FPE telescope temperature (in DN units)
fpe_tscope_temp_dn = housekeeping_data['ICIE__FPE_TSCOPE_TEMP'].values

# Extract housekeeping timestamps
hk_timestamps = pd.to_datetime(housekeeping_data['PACKET_ICIE_TIME'].values)

print(f"Housekeeping temperature data loaded:")
print(f"  Number of samples: {len(hk_timestamps):,}")
print(f"  Start time: {hk_timestamps[0]}")
print(f"  End time: {hk_timestamps[-1]}")
print(f"  Duration: {(hk_timestamps[-1] - hk_timestamps[0]).total_seconds():.1f} seconds")

# Calculate sample rate
if len(hk_timestamps) > 1:
    time_diffs = np.diff(hk_timestamps).astype('timedelta64[ms]').astype(float) / 1000.0
    avg_sample_period = np.mean(time_diffs)
    sample_rate = 1.0 / avg_sample_period if avg_sample_period > 0 else 0
    print(f"  Average sample period: {avg_sample_period:.3f} seconds")
    print(f"  Approximate sample rate: {sample_rate:.3f} Hz")

print(f"\nFPE Telescope Temperature (DN) statistics:")
print(f"  mean={np.mean(fpe_tscope_temp_dn):.2f}, std={np.std(fpe_tscope_temp_dn):.2f}")
print(f"  min={np.min(fpe_tscope_temp_dn):.2f}, max={np.max(fpe_tscope_temp_dn):.2f}")

# Create DataFrame for housekeeping data
df_hk = pd.DataFrame({
    'time': hk_timestamps,
    'fpe_tscope_temp_dn': fpe_tscope_temp_dn,
})
df_hk.set_index('time', inplace=True)

#### Interpolate Temperature to the Radiometer Time Grid

Linear time-interpolation is used to produce a temperature value aligned with every 200 Hz radiometer sample. The interpolated column `temp_dn_interp` is appended directly to the `radiometer_dns` DataFrame so it travels with the DN data through subsequent steps.

In [ ]:
# Interpolate housekeeping temperature to radiometer sample times
print("\nInterpolating temperature to radiometer time grid...")

# Reindex to radiometer times and interpolate
df_temp_interp = df_hk.reindex(
    df_hk.index.union(radiometer_dns.index)
).interpolate(method='time').loc[radiometer_dns.index]

# Add interpolated temperature to the radiometer DataFrame
radiometer_dns['temp_dn_interp'] = df_temp_interp['fpe_tscope_temp_dn'].values
print("Complete!")
print(radiometer_dns)

The first plot shows the raw housekeeping temperature at its native ~1 Hz cadence. The combined plot overlays the interpolated temperature with all four radiometer DN channels on a shared time axis, confirming that the interpolation covers the full observation window and that temperature variation is smooth relative to the DN signal.

In [ ]:
# Plot the full temperature time series
fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('FPE Telescope Temperature DN', fontsize=14, fontweight='bold')

ax.plot(df_hk.index, df_hk['fpe_tscope_temp_dn'], color='#e74c3c', linewidth=1, alpha=0.8)
ax.set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
ax.set_ylabel('Temperature DN', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

# Add statistics text
mean_val = df_hk['fpe_tscope_temp_dn'].mean()
std_val = df_hk['fpe_tscope_temp_dn'].std()
min_val = df_hk['fpe_tscope_temp_dn'].min()
max_val = df_hk['fpe_tscope_temp_dn'].max()
ax.text(0.02, 0.97, f'μ={mean_val:.2f}, σ={std_val:.3f}\nmin={min_val:.2f}, max={max_val:.2f}\nn={len(df_hk):,}',
        transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {len(df_hk):,} housekeeping samples")

# Create combined plot with temperature and radiometer data
df_plot = radiometer_dns.iloc[:n_samples_plot]

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f'Combined Temperature and Radiometer DN',
             fontsize=14, fontweight='bold')

channels = ['ch0', 'ch1', 'ch2', 'ch3']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Temperature plot (top)
ax = axes[0]
ax.plot(df_plot.index, df_plot['temp_dn_interp'],
        color='#e74c3c', linewidth=1.5, alpha=0.8)
ax.set_ylabel('FPE Temp\n(DN)', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3)
mean_val = df_plot['temp_dn_interp'].mean()
std_val = df_plot['temp_dn_interp'].std()
ax.text(0.02, 0.95, f'μ={mean_val:.2f}, σ={std_val:.3f}',
        transform=ax.transAxes, fontsize=9,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Radiometer channels (bottom 4 plots)
for i, (ch, color) in enumerate(zip(channels, colors)):
    ax = axes[i+1]
    ax.plot(df_plot.index, df_plot[f'dn_{ch}'],
            color=color, linewidth=0.8, alpha=0.8)
    ax.set_ylabel(f'{channel_names[i]}\n(DN)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

    mean_val = df_plot[f'dn_{ch}'].mean()
    std_val = df_plot[f'dn_{ch}'].std()
    ax.text(0.02, 0.95, f'μ={mean_val:.1f}, σ={std_val:.2f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[-1].set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
axes[-1].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {n_samples_plot:,} samples ({n_samples_plot/200:.1f} seconds)")

---
## Step 3 — Gain Calibration

The raw DN signal contains frequency-dependent gain non-uniformities introduced by the instrument's analog electronics. Gain calibration corrects for this by applying a frequency-domain filter derived from ground measurements. For more information on how this filter is generated in production, see the [Gain and Noise Event Design Document](https://lasp.colorado.edu/galaxy/x/A6jUEg).

### A. Build the Transfer Function

The transfer function describes how the instrument's gain varies with frequency. It is computed over the full real-FFT frequency grid for the 200 Hz signal using `gain_calibration.get_ground_cal_response_function()`, which reads the ground calibration coefficients bundled with the `libera_rad` package. The plot shows gain coefficient vs. frequency — values away from 1.0 indicate frequency bands where the electronics introduce gain error that must be corrected.

In [ ]:
sampling_rate = 200.0  # Hz
n_samples = len(rad_ch0)

# Generate frequency array for the transfer function
freqs = rfftfreq(n_samples, 1/sampling_rate)
transfer_function = gain_calibration.get_ground_cal_response_function(freqs)

# Plot freqs vs transfer function
plt.plot(freqs, transfer_function)
plt.title("Transfer Function")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Gain Coefficient")
plt.tight_layout()
plt.show()

### B. Apply Calibration to Each Channel

`gain_calibration.apply_gain_calibration()` transforms each channel into the frequency domain via FFT, divides by the transfer function to flatten the gain response, then transforms back to the time domain via inverse FFT. This is applied independently to all four channels, producing the `rad_chN_calibrated` arrays at 200 Hz.

In [ ]:
# Apply calibration to each channel
print("\nApplying gain calibration to radiometer channels...")

rad_ch0_calibrated = gain_calibration.apply_gain_calibration(rad_ch0, transfer_function, n_samples)
rad_ch1_calibrated = gain_calibration.apply_gain_calibration(rad_ch1, transfer_function, n_samples)
rad_ch2_calibrated = gain_calibration.apply_gain_calibration(rad_ch2, transfer_function, n_samples)
rad_ch3_calibrated = gain_calibration.apply_gain_calibration(rad_ch3, transfer_function, n_samples)

print("Calibration complete!")

# Print statistics comparison
print("\nCalibration Statistics Comparison:")
print("Shortwave (Channel 0):")
print(f"  Uncalibrated - Mean: {np.mean(rad_ch0):.2f}, Std: {np.std(rad_ch0):.2f}")
print(f"  Calibrated   - Mean: {np.mean(rad_ch0_calibrated):.2f}, Std: {np.std(rad_ch0_calibrated):.2f}")
print("Total (Channel 1):")
print(f"  Uncalibrated - Mean: {np.mean(rad_ch1):.2f}, Std: {np.std(rad_ch1):.2f}")
print(f"  Calibrated   - Mean: {np.mean(rad_ch1_calibrated):.2f}, Std: {np.std(rad_ch1_calibrated):.2f}")
print("Longwave (Channel 2):")
print(f"  Uncalibrated - Mean: {np.mean(rad_ch2):.2f}, Std: {np.std(rad_ch2):.2f}")
print(f"  Calibrated   - Mean: {np.mean(rad_ch2_calibrated):.2f}, Std: {np.std(rad_ch2_calibrated):.2f}")
print("Split Shortwave (Channel 3):")
print(f"  Uncalibrated - Mean: {np.mean(rad_ch3):.2f}, Std: {np.std(rad_ch3):.2f}")
print(f"  Calibrated   - Mean: {np.mean(rad_ch3_calibrated):.2f}, Std: {np.std(rad_ch3_calibrated):.2f}")

The side-by-side comparison below shows 2 seconds of each channel before and after calibration. The mean should remain nearly unchanged while the standard deviation decreases, reflecting the removal of correlated gain-induced noise.

In [ ]:
# Plot comparison of uncalibrated vs calibrated signals in time domain
fig, axes = plt.subplots(4, 2, figsize=(16, 12))

# Select time window for detailed view
plot_duration_compare = 2.0  # seconds
plot_mask_compare = (radiometer_timestamps <= radiometer_timestamps[0] + pd.Timedelta(seconds=plot_duration_compare))
time_seconds_compare = (radiometer_timestamps[plot_mask_compare] - radiometer_timestamps[0]).total_seconds()

channels_uncalib = [rad_ch0, rad_ch1, rad_ch2, rad_ch3]
channels_calib = [rad_ch0_calibrated, rad_ch1_calibrated, rad_ch2_calibrated, rad_ch3_calibrated]
colors = ['blue', 'green', 'red', 'purple']

for i, (name, color) in enumerate(zip(channel_names, colors)):
    # Plot uncalibrated signal
    axes[i, 0].plot(time_seconds_compare, channels_uncalib[i][plot_mask_compare],
                    color=color, linewidth=0.5, alpha=0.8)
    axes[i, 0].set_ylabel('DN Value')
    axes[i, 0].set_title(f'{name} - Uncalibrated (200 Hz)')
    axes[i, 0].grid(True, alpha=0.3)

    # Add statistics
    mean_uncalib = np.mean(channels_uncalib[i][plot_mask_compare])
    std_uncalib = np.std(channels_uncalib[i][plot_mask_compare])
    axes[i, 0].text(0.02, 0.95, f'Mean: {mean_uncalib:.1f}\nStd: {std_uncalib:.1f}',
                    transform=axes[i, 0].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

    # Plot calibrated signal
    axes[i, 1].plot(time_seconds_compare, channels_calib[i][plot_mask_compare],
                    color=color, linewidth=0.5, alpha=0.8)
    axes[i, 1].set_ylabel('Calibrated Value')
    axes[i, 1].set_title(f'{name} - Calibrated (200 Hz)')
    axes[i, 1].grid(True, alpha=0.3)

    # Add statistics
    mean_calib = np.mean(channels_calib[i][plot_mask_compare])
    std_calib = np.std(channels_calib[i][plot_mask_compare])
    axes[i, 1].text(0.02, 0.95, f'Mean: {mean_calib:.1f}\nStd: {std_calib:.1f}',
                    transform=axes[i, 1].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

axes[-1, 0].set_xlabel('Time (seconds)')
axes[-1, 1].set_xlabel('Time (seconds)')

fig.suptitle('Comparison: Uncalibrated vs Calibrated Signals (Time Domain, 200 Hz)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Plotted {np.sum(plot_mask_compare):,} samples for comparison")

---
## Step 4 — Downsample from 200 Hz to 100 Hz

The L1B product specification requires radiance at **100 Hz**. `gain_calibration.downsample_libera_signal()` downsamples the data from 200 to 100 Hz. Both the calibrated and (for validation purposes) the uncalibrated signals are downsampled, and a corresponding 100 Hz timestamp array is constructed by taking every other sample from the original 200 Hz timestamps.

In [ ]:
# Downsample both uncalibrated and calibrated signals from 200 Hz to 100 Hz
print("Downsampling signals from 200 Hz to 100 Hz...")

# Downsample uncalibrated signals
rad_ch0_100hz = gain_calibration.downsample_libera_signal(rad_ch0, from_rate=200.0, to_rate=100.0)
rad_ch1_100hz = gain_calibration.downsample_libera_signal(rad_ch1, from_rate=200.0, to_rate=100.0)
rad_ch2_100hz = gain_calibration.downsample_libera_signal(rad_ch2, from_rate=200.0, to_rate=100.0)
rad_ch3_100hz = gain_calibration.downsample_libera_signal(rad_ch3, from_rate=200.0, to_rate=100.0)

# Downsample calibrated signals
rad_ch0_calibrated_100hz = gain_calibration.downsample_libera_signal(rad_ch0_calibrated, from_rate=200.0, to_rate=100.0)
rad_ch1_calibrated_100hz = gain_calibration.downsample_libera_signal(rad_ch1_calibrated, from_rate=200.0, to_rate=100.0)
rad_ch2_calibrated_100hz = gain_calibration.downsample_libera_signal(rad_ch2_calibrated, from_rate=200.0, to_rate=100.0)
rad_ch3_calibrated_100hz = gain_calibration.downsample_libera_signal(rad_ch3_calibrated, from_rate=200.0, to_rate=100.0)

# Create time array for 100 Hz data
radiometer_timestamps_100hz = radiometer_timestamps[::2][:len(rad_ch0_100hz)]  # Every other sample

print(f"Downsampling complete!")
print(f"  Original samples (200 Hz): {len(rad_ch0):,}")
print(f"  Downsampled samples (100 Hz): {len(rad_ch0_100hz):,}")



The 4×2 grid below compares uncalibrated and calibrated signals at 100 Hz over a 2-second window. The calibrated traces should show lower noise than the uncalibrated ones, consistent with the gain correction applied in Step 3. The downsampling itself should not change the signal mean appreciably.

In [ ]:
# Compare 100 Hz downsampled signals (uncalibrated vs calibrated)
fig, axes = plt.subplots(4, 2, figsize=(16, 12))

# Time array for 100 Hz data
time_100hz = (radiometer_timestamps_100hz - radiometer_timestamps_100hz[0]).total_seconds()
plot_mask_100hz = time_100hz <= plot_duration_compare

channels_uncalib_100hz = [rad_ch0_100hz, rad_ch1_100hz, rad_ch2_100hz, rad_ch3_100hz]
channels_calib_100hz = [rad_ch0_calibrated_100hz, rad_ch1_calibrated_100hz,
                        rad_ch2_calibrated_100hz, rad_ch3_calibrated_100hz]

for i, (name, color) in enumerate(zip(channel_names, colors)):
    # Plot uncalibrated 100 Hz signal
    axes[i, 0].plot(time_100hz[plot_mask_100hz], channels_uncalib_100hz[i][plot_mask_100hz],
                    color=color, linewidth=0.8, alpha=0.8)
    axes[i, 0].set_ylabel('DN Value')
    axes[i, 0].set_title(f'{name} - Uncalibrated (100 Hz)')
    axes[i, 0].grid(True, alpha=0.3)

    # Add statistics
    mean_uncalib_100 = np.mean(channels_uncalib_100hz[i][plot_mask_100hz])
    std_uncalib_100 = np.std(channels_uncalib_100hz[i][plot_mask_100hz])
    axes[i, 0].text(0.02, 0.95, f'Mean: {mean_uncalib_100:.1f}\nStd: {std_uncalib_100:.1f}',
                    transform=axes[i, 0].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

    # Plot calibrated 100 Hz signal
    axes[i, 1].plot(time_100hz[plot_mask_100hz], channels_calib_100hz[i][plot_mask_100hz],
                    color=color, linewidth=0.8, alpha=0.8)
    axes[i, 1].set_ylabel('Calibrated Value')
    axes[i, 1].set_title(f'{name} - Calibrated (100 Hz)')
    axes[i, 1].grid(True, alpha=0.3)

    # Add statistics
    mean_calib_100 = np.mean(channels_calib_100hz[i][plot_mask_100hz])
    std_calib_100 = np.std(channels_calib_100hz[i][plot_mask_100hz])
    axes[i, 1].text(0.02, 0.95, f'Mean: {mean_calib_100:.1f}\nStd: {std_calib_100:.1f}',
                    transform=axes[i, 1].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)

axes[-1, 0].set_xlabel('Time (seconds)')
axes[-1, 1].set_xlabel('Time (seconds)')

fig.suptitle('Comparison: Uncalibrated vs Calibrated Signals (100 Hz Downsampled)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Plotted {np.sum(plot_mask_100hz):,} samples at 100 Hz for comparison")

---
## Step 5 — Convert DN to Radiance

The final algorithmic step converts calibrated DN values into physical radiance (W m⁻² sr⁻¹) using the instrument's radiometric calibration coefficients.

`radiance.interpolate_temperatures()` re-interpolates the housekeeping temperature onto the 100 Hz time grid (required because the grid changed after downsampling). `radiance.calculate_radiances()` then applies the temperature-dependent conversion coefficients to each channel independently, producing the `radiance_100hz` dictionary keyed by channel name (`sw`, `total`, `lw`, `ssw`).

In [ ]:
calibration_data = radiance._load_calibration_data()

print("Converting DN to radiance in W m^-2 sr^-1...")

data_by_channel = {
    "sw": rad_ch0_calibrated_100hz,
    "total": rad_ch1_calibrated_100hz,
    "lw": rad_ch2_calibrated_100hz,
    "ssw": rad_ch3_calibrated_100hz
}

# interpolate temperatures
interpolated_temps = radiance.interpolate_temperatures(radiometer_timestamps_100hz, nom_hk_data=housekeeping_data)

# Convert each channel to radiance
radiance_100hz = radiance.calculate_radiances(data_by_channel, interpolated_temps)
print("Radiance conversion complete!")
for key, value in data_by_channel.items():
    print(f"    Channel {key}: mean = {value.mean()} W m^-2 sr^-1, {value.size} values")

### Plotting Utility Functions

The cell below defines reusable helper functions (`plot_channels_stacked`, `plot_channels_overlay`, `plot_processing_pipeline`, `plot_pipeline_grid`) used to visualise the radiance output and processing pipeline in the sections that follow.

  1. plot_channels_stacked   – one subplot per channel, shared x-axis
  2. plot_channels_overlay   – all channels on a single axes
  3. plot_processing_pipeline – four processing stages for one channel

In [ ]:
# Shared defaults

_DEFAULT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
_STAT_BOX = dict(boxstyle="round", facecolor="white", alpha=0.7)
_TIME_FMT = DateFormatter("%H:%M:%S")


def _add_stat_annotation(ax, data, precision=2, transform=None, bbox=None):
    """Place a μ / σ text box in the upper-left corner of *ax*."""
    mean_val = np.mean(data)
    std_val = np.std(data)
    fmt = f".{precision}f"
    text = f"μ={mean_val:{fmt}}, σ={std_val:{fmt}}"
    ax.text(
        0.02, 0.95, text,
        transform=transform or ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=bbox or _STAT_BOX,
    )


def plot_channels_stacked(
    time_index,
    channel_data,
    channel_labels=None,
    ylabel_unit="DN",
    title="Radiometer Channel Time Series",
    colors=None,
    time_fmt=None,
    figsize=(14, 10),
    stat_precision=2,
):
    """
    Plot one subplot per channel, stacked vertically with a shared x-axis.

    Parameters
    ----------
    time_index : array-like
        Timestamps (datetime-like) used as the x-axis.
    channel_data : list of array-like
        One array per channel, in the same order as *channel_labels*.
    channel_labels : list of str, optional
        Display labels for each channel (e.g. ["SW", "Total", "LW", "SSW"]).
        Defaults to ["CH0", "CH1", ...].
    ylabel_unit : str
        Unit string appended to each y-axis label (e.g. "DN" or "W m⁻² sr⁻¹").
    title : str
        Figure title.
    colors : list of str, optional
        Line colors for each channel. Defaults to the standard 4-color palette.
    time_fmt : matplotlib DateFormatter, optional
        Formatter for the x-axis. Defaults to "%H:%M:%S".
    figsize : tuple
        Figure size passed to plt.subplots.
    stat_precision : int
        Decimal places in the μ / σ annotation.

    Returns
    -------
    fig, axes : matplotlib Figure and array of Axes
    """
    n = len(channel_data)
    if channel_labels is None:
        channel_labels = [f"CH{i}" for i in range(n)]
    if colors is None:
        colors = (_DEFAULT_COLORS * ((n // len(_DEFAULT_COLORS)) + 1))[:n]
    if time_fmt is None:
        time_fmt = _TIME_FMT

    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=True)
    if n == 1:
        axes = [axes]

    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i, (ax, data, label, color) in enumerate(
        zip(axes, channel_data, channel_labels, colors)
    ):
        ax.plot(time_index, data, color=color, linewidth=0.8, alpha=0.8)
        ax.set_ylabel(f"{label}\n({ylabel_unit})", fontsize=10, fontweight="bold")
        ax.grid(True, alpha=0.3)
        _add_stat_annotation(ax, data, precision=stat_precision)

    axes[-1].set_xlabel("Time (UTC)", fontsize=11, fontweight="bold")
    axes[-1].xaxis.set_major_formatter(time_fmt)
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig, axes


def plot_channels_overlay(
    time_index,
    channel_data,
    channel_labels=None,
    ylabel="Value",
    title="All Radiometer Channels",
    colors=None,
    time_fmt=None,
    figsize=(14, 6),
):
    """
    Plot all channels on a single axes for direct comparison.

    Parameters
    ----------
    time_index : array-like
        Timestamps used as the x-axis.
    channel_data : list of array-like
        One array per channel.
    channel_labels : list of str, optional
        Legend labels for each channel.
    ylabel : str
        Y-axis label.
    title : str
        Figure title.
    colors : list of str, optional
        Line colors for each channel.
    time_fmt : matplotlib DateFormatter, optional
        Formatter for the x-axis.
    figsize : tuple
        Figure size.

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    """
    n = len(channel_data)
    if channel_labels is None:
        channel_labels = [f"CH{i}" for i in range(n)]
    if colors is None:
        colors = (_DEFAULT_COLORS * ((n // len(_DEFAULT_COLORS)) + 1))[:n]
    if time_fmt is None:
        time_fmt = _TIME_FMT

    fig, ax = plt.subplots(figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for data, label, color in zip(channel_data, channel_labels, colors):
        ax.plot(time_index, data, color=color, linewidth=1.0, alpha=0.8, label=label)

    ax.set_xlabel("Time (UTC)", fontsize=11, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    ax.legend(loc="best", fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(time_fmt)
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig, ax


_PIPELINE_STAGE_DEFAULTS = [
    {"label": "Raw DN\n(200 Hz)",         "color": "b",      "bbox_color": "lightblue"},
    {"label": "Calibrated DN\n(200 Hz)",  "color": "g",      "bbox_color": "lightgreen"},
    {"label": "Downsampled DN\n(100 Hz)", "color": "orange", "bbox_color": "moccasin"},
    {"label": "Radiance\n(W m^-2 sr^-1)\n(100 Hz)", "color": "r", "bbox_color": "lightcoral"},
]


def plot_processing_pipeline(
    stages,
    channel_name="Channel",
    stage_meta=None,
    time_fmt=None,
    figsize=(16, 12),
):
    """
    Plot each processing stage for a single radiometer channel as stacked
    subplots with a shared x-axis.

    Parameters
    ----------
    stages : list of dict
        Each dict must contain:
          - "time"  : array-like of timestamps or elapsed seconds
          - "data"  : array-like of signal values
        Optional keys:
          - "ylabel" : y-axis label (overrides *stage_meta*)
          - "color"  : line color (overrides *stage_meta*)
          - "bbox_color" : annotation box color (overrides *stage_meta*)
          - "stat_fmt"   : format string for μ/σ (e.g. ".2f" or ".6f")

    channel_name : str
        Channel identifier used in the figure title
        (e.g. "Channel 0 (Shortwave)").

    stage_meta : list of dict, optional
        Per-stage display overrides with the same keys as the optional *stages*
        keys. If None, uses the 4-stage defaults (raw DN → calibrated DN →
        downsampled DN → radiance).

    time_fmt : matplotlib DateFormatter, optional
        Formatter for the x-axis of the last subplot.
        Defaults to "%H:%M:%S.%f".

    figsize : tuple
        Figure size.

    Returns
    -------
    fig, axes : matplotlib Figure and array of Axes

    Examples
    --------
    >>> stages = [
    ...     {"time": time_200hz, "data": ch0_raw_dn},
    ...     {"time": time_200hz, "data": ch0_calibrated_dn},
    ...     {"time": time_100hz, "data": ch0_downsampled_dn},
    ...     {"time": time_100hz, "data": ch0_radiance, "stat_fmt": ".6f"},
    ... ]
    >>> fig, axes = plot_processing_pipeline(stages, channel_name="Channel 0 (Shortwave)")
    >>> plt.show()
    """
    n = len(stages)
    if stage_meta is None:
        stage_meta = (_PIPELINE_STAGE_DEFAULTS * ((n // len(_PIPELINE_STAGE_DEFAULTS)) + 1))[:n]
    if time_fmt is None:
        time_fmt = DateFormatter("%H:%M:%S.%f")

    fig, axes = plt.subplots(n, 1, figsize=figsize, sharex=False)
    if n == 1:
        axes = [axes]

    fig.suptitle(
        f"{channel_name} – Complete L1B Processing Pipeline",
        fontsize=16, fontweight="bold",
    )

    for i, (ax, stage, meta) in enumerate(zip(axes, stages, stage_meta)):
        time_arr = stage["time"]
        data = stage["data"]
        color = stage.get("color", meta.get("color", "steelblue"))
        ylabel = stage.get("ylabel", meta.get("label", f"Stage {i}"))
        bbox_color = stage.get("bbox_color", meta.get("bbox_color", "white"))
        stat_fmt_str = stage.get("stat_fmt", ".2f" if i < 3 else ".6f")
        precision = len(stat_fmt_str.strip(".f"))

        lw = 0.8 if i < 2 else 1.0
        ax.plot(time_arr, data, color=color, linewidth=lw, alpha=0.8)
        ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
        ax.grid(True, alpha=0.3)

        mean_val = np.mean(data)
        std_val = np.std(data)
        unit = "W m⁻² sr⁻¹" if i == n - 1 else "DN"
        stat_text = f"Mean: {mean_val:{stat_fmt_str}} {unit}, Std: {std_val:{stat_fmt_str}}"
        ax.text(
            0.02, 0.95, stat_text,
            transform=ax.transAxes, fontsize=10,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor=bbox_color, alpha=0.8),
        )

        # Share the same x-axis range across all subplots by syncing limits
        # after the last subplot is drawn (sharex=False lets each stage use
        # its own time array, which may differ in length between 200 Hz and
        # 100 Hz stages).

    axes[-1].set_xlabel("Time (UTC)", fontsize=11, fontweight="bold")
    axes[-1].xaxis.set_major_formatter(time_fmt)
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig, axes


def plot_pipeline_grid(
    channels_data,
    time_200hz,
    time_100hz,
    stage_keys=("raw", "calibrated", "downsampled", "radiance"),
    stage_names=None,
    figsize=(20, 14),
):
    """
    Produce the 4×4 grid showing every processing stage for every channel.

    Parameters
    ----------
    channels_data : list of dict
        Each dict must have keys matching *stage_keys* plus:
          - "name"  : str, channel display name
          - "color" : str, line color
    time_200hz : array-like
        Timestamps for 200 Hz stages (raw, calibrated).
    time_100hz : array-like
        Timestamps for 100 Hz stages (downsampled, radiance).
    stage_keys : tuple of str
        Keys into each channel dict for the four processing stages.
    stage_names : list of str, optional
        Y-axis labels for each stage row.
    figsize : tuple
        Figure size.

    Returns
    -------
    fig, axes : matplotlib Figure and 2-D array of Axes
    """
    if stage_names is None:
        stage_names = [
            "Raw DN\n(200 Hz)",
            "Calibrated DN\n(200 Hz)",
            "Downsampled DN\n(100 Hz)",
            "Radiance\n(W m$^{-2}$ sr$^{-1}$)\n(100 Hz)",
        ]

    n_stages = len(stage_keys)
    n_channels = len(channels_data)

    fig, axes = plt.subplots(n_stages, n_channels, figsize=figsize, sharex=False)
    fig.suptitle(
        "Complete L1B Processing Pipeline – All Channels",
        fontsize=18, fontweight="bold",
    )

    high_hz_keys = set(stage_keys[:2])   # raw, calibrated → 200 Hz

    for stage_idx, (stage_key, stage_name) in enumerate(zip(stage_keys, stage_names)):
        time_array = time_200hz if stage_key in high_hz_keys else time_100hz
        is_radiance = stage_key == stage_keys[-1]

        for ch_idx, ch in enumerate(channels_data):
            ax = axes[stage_idx, ch_idx]
            data = ch[stage_key]

            lw = 0.8 if stage_key in high_hz_keys else 1.0
            ax.plot(time_array, data, color=ch["color"], linewidth=lw, alpha=0.8)

            if stage_idx == 0:
                ax.set_title(ch["name"], fontsize=12, fontweight="bold")
            if ch_idx == 0:
                ax.set_ylabel(stage_name, fontsize=10, fontweight="bold")

            ax.grid(True, alpha=0.3)

            mean_val = np.mean(data)
            std_val = np.std(data)
            fmt = ".4f\n±{:.6f}" if is_radiance else ".1f\n±{:.2f}"
            label = f"{mean_val:.4f}\n±{std_val:.6f}" if is_radiance \
                else f"{mean_val:.1f}\n±{std_val:.2f}"
            ax.text(
                0.98, 0.95, label,
                transform=ax.transAxes, fontsize=8,
                verticalalignment="top", horizontalalignment="right",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
            )

            if stage_idx == n_stages - 1:
                ax.xaxis.set_major_formatter(DateFormatter("%H:%M:%S"))
                plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
                ax.set_xlabel("Time (UTC)", fontsize=9)

    plt.tight_layout()
    return fig, axes

### Radiance Output — Visualisation and Statistics

The radiance values are collected into a time-indexed DataFrame and plotted two ways: stacked subplots (one per channel) to inspect individual channel behavior, and an overlay plot for direct amplitude comparison across channels. The statistical summary printed below each plot reports mean, standard deviation, coefficient of variation, and signal-to-noise ratio for every channel, providing a quick sanity check on the calibration quality.

In [ ]:
# Create DataFrame with radiance values
df_radiance = pd.DataFrame({
    'time': radiometer_timestamps_100hz,
    'radiance_ch0': radiance_100hz["sw"],
    'radiance_ch1': radiance_100hz["total"],
    'radiance_ch2': radiance_100hz["lw"],
    'radiance_ch3': radiance_100hz["ssw"]
}).set_index('time')

# Plot first 10 seconds of radiance data

#n_samples_plot = min(1000, len(df_radiance))  # 1000 samples at 100 Hz = 10 seconds
n_samples_plot = len(df_radiance)
df_plot = df_radiance.iloc[:n_samples_plot]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Calibrated Radiometer Data - Radiance (100 Hz)', fontsize=14, fontweight='bold')

channels = ['ch0', 'ch1', 'ch2', 'ch3']
channel_labels = [
    "sw",
    "total",
    "lw",
    "ssw"
]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, (ax, ch, label, color) in enumerate(zip(axes, channels, channel_labels, colors)):
    ax.plot(df_plot.index, df_plot[f'radiance_{ch}'],
            color=color, linewidth=0.8, alpha=0.8, label=label)
    ax.set_ylabel(f'{label.upper()}\n(W m^-2 sr^-1)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)

    mean_val = df_plot[f'radiance_{ch}'].mean()
    std_val = df_plot[f'radiance_{ch}'].std()
    ax.text(0.02, 0.95, f'μ={mean_val:.4f}, σ={std_val:.6f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[-1].set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
axes[-1].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {n_samples_plot} samples at 100 Hz ({n_samples_plot/100:.1f} seconds)")

fig, ax = plot_channels_overlay(
    df_plot.index,
    [df_plot[f"radiance_{ch}"] for ch in ["ch0","ch1","ch2","ch3"]],
    channel_labels=["SW", "Total", "LW", "SSW"],
    ylabel="Radiance (W m^-2 sr^-1)",
    title="All Channels – Radiance Comparison (100 Hz)",
)


print("Statistical Comparison: Calibrated DN vs Radiance")
print("=" * 70)

for i, ch in enumerate(['sw', 'total', 'lw', 'ssw']):
    rad = radiance_100hz[ch]
    if i == 0:
        dn = rad_ch0_calibrated_100hz
    elif i == 1:
        dn = rad_ch1_calibrated_100hz
    elif i == 2:
        dn = rad_ch2_calibrated_100hz
    else:
        dn = rad_ch3_calibrated_100hz


    print(f"\{ch.upper()}:")
    print(f"  DN Statistics:")
    print(f"    Mean: {np.mean(dn):.3f} DN")
    print(f"    Std:  {np.std(dn):.4f} DN")
    print(f"    CV:   {(np.std(dn)/np.mean(dn)*100):.4f}%")
    print(f"  Radiance Statistics:")
    print(f"    Mean: {np.mean(rad):.6f} W m^-2 sr^-1")
    print(f"    Std:  {np.std(rad):.8f} W m^-2 sr^-1")
    print(f"    CV:   {(np.std(rad)/np.mean(rad)*100):.4f}%")
    print(f"  Signal-to-Noise Ratio: {np.mean(rad)/np.std(rad):.1f}")

In [ ]:
# Create DataFrame with radiance values
df_radiance = pd.DataFrame({
    'time': radiometer_timestamps_100hz,
    'radiance_ch0': radiance_100hz["sw"],
    'radiance_ch1': radiance_100hz["total"],
    'radiance_ch2': radiance_100hz["lw"],
    'radiance_ch3': radiance_100hz["ssw"]
}).set_index('time')

# Plot first 10 seconds of radiance data

#n_samples_plot = min(1000, len(df_radiance))  # 1000 samples at 100 Hz = 10 seconds
n_samples_plot = len(df_radiance)
df_plot = df_radiance.iloc[:n_samples_plot]

fig, axes = plot_channels_stacked(
    df_plot.index,
    [df_plot[f"radiance_{ch}"] for ch in ["ch0","ch1","ch2","ch3"]],
    channel_labels=["SW", "TOTAL", "LW", "SSW"],
    ylabel_unit="W m^-2 sr^-1",
    title="Calibrated Radiometer Data - Radiance (100 Hz)",
)

print(f"Plotted {n_samples_plot} samples at 100 Hz ({n_samples_plot/100:.1f} seconds)")

fig, ax = plot_channels_overlay(
    df_plot.index,
    [df_plot[f"radiance_{ch}"] for ch in ["ch0","ch1","ch2","ch3"]],
    channel_labels=["SW", "TOTAL", "LW", "SSW"],
    ylabel="Radiance (W m^-2 sr^-1)",
    title="All Channels – Radiance Comparison (100 Hz)",
)

print(f"Plotted {n_samples_plot} samples at 100 Hz ({n_samples_plot/100:.1f} seconds)")


print("Statistical Comparison: Calibrated DN vs Radiance")
print("=" * 70)

for i, ch in enumerate(['sw', 'total', 'lw', 'ssw']):
    rad = radiance_100hz[ch]
    if i == 0:
        dn = rad_ch0_calibrated_100hz
    elif i == 1:
        dn = rad_ch1_calibrated_100hz
    elif i == 2:
        dn = rad_ch2_calibrated_100hz
    else:
        dn = rad_ch3_calibrated_100hz


    print(f"\{ch.upper()}:")
    print(f"  DN Statistics:")
    print(f"    Mean: {np.mean(dn):.3f} DN")
    print(f"    Std:  {np.std(dn):.4f} DN")
    print(f"    CV:   {(np.std(dn)/np.mean(dn)*100):.4f}%")
    print(f"  Radiance Statistics:")
    print(f"    Mean: {np.mean(rad):.6f} W m^-2 sr^-1")
    print(f"    Std:  {np.std(rad):.8f} W m^-2 sr^-1")
    print(f"    CV:   {(np.std(rad)/np.mean(rad)*100):.4f}%")
    print(f"  Signal-to-Noise Ratio: {np.mean(rad)/np.std(rad):.1f}")

---
## Processing Pipeline Visualisation

The cells below show the complete transformation from raw DN to radiance for a short (2-second) window of data. Each channel is displayed as a 4-panel figure stepping through: raw DN (200 Hz) → gain-calibrated DN (200 Hz) → downsampled DN (100 Hz) → radiance (100 Hz). This makes it straightforward to see the effect of each processing step on the signal level and noise.

The time window is set to the first 2 seconds (400 samples at 200 Hz / 200 samples at 100 Hz) to keep the figures legible.

In [ ]:
# Select time window for comparison (first 2 seconds = 400 samples at 200 Hz)
n_samples_200hz = 400
n_samples_100hz = 200
# Get timestamps
time_200hz = radiometer_timestamps[:n_samples_200hz]
time_100hz = radiometer_timestamps_100hz[:n_samples_100hz]

The four pipeline figures below are produced by `plot_processing_pipeline()` — one figure per channel. Annotation boxes in each subplot report the mean and standard deviation at that processing stage.

In [ ]:
# Get data for each processing stage
ch0_raw_dn = rad_ch0[:n_samples_200hz]
ch0_calibrated_dn = rad_ch0_calibrated[:n_samples_200hz]
ch0_downsampled_dn = rad_ch0_calibrated_100hz[:n_samples_100hz]
ch0_radiance = radiance_100hz['sw'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch0_raw_dn},
        {"time": time_200hz, "data": ch0_calibrated_dn},
        {"time": time_100hz, "data": ch0_downsampled_dn},
        {"time": time_100hz, "data": ch0_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Shortwave (Channel 0)",
)

print(f"Channel 0 (Shortwave) - Processing stages comparison:")
print(f"  Time window: {n_samples_200hz/200:.1f} seconds")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch0_raw_dn):.3f} to {np.std(ch0_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch0_radiance):.6f} ± {np.std(ch0_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch1_raw_dn = rad_ch1[:n_samples_200hz]
ch1_calibrated_dn = rad_ch1_calibrated[:n_samples_200hz]
ch1_downsampled_dn = rad_ch1_calibrated_100hz[:n_samples_100hz]
ch1_radiance = radiance_100hz['total'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch1_raw_dn},
        {"time": time_200hz, "data": ch1_calibrated_dn},
        {"time": time_100hz, "data": ch1_downsampled_dn},
        {"time": time_100hz, "data": ch1_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Total (Channel 1)",
)

print(f"Channel 1 (Total) - Processing stages comparison:")
print(f"  Time window: {n_samples_200hz/200:.1f} seconds")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch1_raw_dn):.3f} to {np.std(ch1_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch1_radiance):.6f} ± {np.std(ch1_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch2_raw_dn = rad_ch2[:n_samples_200hz]
ch2_calibrated_dn = rad_ch2_calibrated[:n_samples_200hz]
ch2_downsampled_dn = rad_ch2_calibrated_100hz[:n_samples_100hz]
ch2_radiance = radiance_100hz['lw'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch2_raw_dn},
        {"time": time_200hz, "data": ch2_calibrated_dn},
        {"time": time_100hz, "data": ch2_downsampled_dn},
        {"time": time_100hz, "data": ch2_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Longwave (Channel 2)",
)

print(f"Channel 2 (Longwave) - Processing stages comparison:")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch2_raw_dn):.3f} to {np.std(ch2_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch2_radiance):.6f} ± {np.std(ch2_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch3_raw_dn = rad_ch3[:n_samples_200hz]
ch3_calibrated_dn = rad_ch3_calibrated[:n_samples_200hz]
ch3_downsampled_dn = rad_ch3_calibrated_100hz[:n_samples_100hz]
ch3_radiance = radiance_100hz['ssw'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch3_raw_dn},
        {"time": time_200hz, "data": ch3_calibrated_dn},
        {"time": time_100hz, "data": ch3_downsampled_dn},
        {"time": time_100hz, "data": ch3_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Split Shortwave (Channel 3)",
)

print(f"Channel 3 (Split Shortwave) - Processing stages comparison:")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch3_raw_dn):.3f} to {np.std(ch3_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch3_radiance):.6f} ± {np.std(ch3_radiance):.8f} W m^-2 sr^-1")

### Summary — All Channels Side by Side

The 4×4 grid below presents the same four processing stages for all four channels simultaneously, making cross-channel comparison straightforward. Rows correspond to processing stages; columns correspond to channels. Mean and standard deviation are annotated in each cell.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex='col')
fig.suptitle('Complete L1B Processing Pipeline - All Channels', fontsize=18, fontweight='bold')

# Define channels
channels_data = [
    {
        'name': 'SW (CH0)',
        'raw': rad_ch0[:n_samples_200hz],
        'calibrated': rad_ch0_calibrated[:n_samples_200hz],
        'downsampled': rad_ch0_calibrated_100hz[:n_samples_100hz],
        'radiance': radiance_100hz['sw'][:n_samples_100hz],
        'color': '#1f77b4'
    },
    {
        'name': 'TOTAL (CH1)',
        'raw': rad_ch1[:n_samples_200hz],
        'calibrated': rad_ch1_calibrated[:n_samples_200hz],
        'downsampled': rad_ch1_calibrated_100hz[:n_samples_100hz],
        'radiance': radiance_100hz['total'][:n_samples_100hz],
        'color': '#ff7f0e'
    },
    {
        'name': 'LW (CH2)',
        'raw': rad_ch2[:n_samples_200hz],
        'calibrated': rad_ch2_calibrated[:n_samples_200hz],
        'downsampled': rad_ch2_calibrated_100hz[:n_samples_100hz],
        'radiance': radiance_100hz['lw'][:n_samples_100hz],
        'color': '#2ca02c'
    },
    {
        'name': 'SSW (CH3)',
        'raw': rad_ch3[:n_samples_200hz],
        'calibrated': rad_ch3_calibrated[:n_samples_200hz],
        'downsampled': rad_ch3_calibrated_100hz[:n_samples_100hz],
        'radiance': radiance_100hz['ssw'][:n_samples_100hz],
        'color': '#d62728'
    }
]


stage_names = ['Raw DN\n(200 Hz)', 'Calibrated DN\n(200 Hz)',
               'Downsampled DN\n(100 Hz)', 'Radiance\n(W m$^-2$ sr$^-1$)\n(100 Hz)']
stage_keys = ['raw', 'calibrated', 'downsampled', 'radiance']

# Plot each stage for each channel
for stage_idx, (stage_name, stage_key) in enumerate(zip(stage_names, stage_keys)):
    for ch_idx, ch_data in enumerate(channels_data):
        ax = axes[stage_idx, ch_idx]

        # Select appropriate time array
        if stage_key in ['raw', 'calibrated']:
            time_array = time_200hz
        else:
            time_array = time_100hz

        # Plot data
        data = ch_data[stage_key]
        ax.plot(time_array, data, color=ch_data['color'],
                linewidth=0.8 if stage_key in ['raw', 'calibrated'] else 1.0,
                alpha=0.8)

        # Set title for top row
        if stage_idx == 0:
            ax.set_title(ch_data['name'], fontsize=12, fontweight='bold')

        # Set ylabel for leftmost column
        if ch_idx == 0:
            ax.set_ylabel(stage_name, fontsize=10, fontweight='bold')

        # Add grid
        ax.grid(True, alpha=0.3)

        # Add statistics
        mean_val = np.mean(data)
        std_val = np.std(data)
        if stage_key == 'radiance':
            label = f'{mean_val:.4f}\n±{std_val:.6f}'
        else:
            label = f'{mean_val:.1f}\n±{std_val:.2f}'

        ax.text(0.98, 0.95, label, transform=ax.transAxes,
                fontsize=8, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

        # Format x-axis for bottom row
        if stage_idx == 3:
            ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
            ax.set_xlabel('Time (UTC)', fontsize=9)

plt.tight_layout()
plt.show()

print("\nProcessing Pipeline Summary:")
print("=" * 80)
for ch_idx, ch_data in enumerate(channels_data):
    print(f"\n{ch_data['name']}:")
    print(f"  Raw DN:                      {np.mean(ch_data['raw']):.2f} ± {np.std(ch_data['raw']):.3f} DN")
    print(f"  Raw DN Data Points:          {len(ch_data['raw'])}")
    print(f"  Calibrated DN:               {np.mean(ch_data['calibrated']):.2f} ± {np.std(ch_data['calibrated']):.3f} DN")
    print(f"  Calibrated DN Data Points:   {len(ch_data['calibrated'])}")
    print(f"  Downsampled DN:              {np.mean(ch_data['downsampled']):.2f} ± {np.std(ch_data['downsampled']):.3f} DN")
    print(f"  Downsampled DN Data Points:  {len(ch_data['downsampled'])}")
    print(f"  Radiance:                    {np.mean(ch_data['radiance']):.6f} ± {np.std(ch_data['radiance']):.8f} W m⁻² sr⁻¹")
    print(f"  Radiance Data Points:        {len(ch_data['radiance'])}")
    print(f"  Noise reduction:             {np.std(ch_data['raw'])/np.std(ch_data['calibrated']):.2f}x")


---
## Tier 1 Validation — Compare Data to Test Output L1B File

This section performs a Tier 1 integration test by loading a reference L1B output file produced by running the full L1B algorithm on AWS, then comparing its radiance values to those computed above. Agreement between the two confirms that this notebook faithfully reproduces the operational algorithm.

The procedure, requirements, and results of the test can be found at [Tier 1 L1B Radiometer Algorithm Test Confluence Page](https://lasp.colorado.edu/galaxy/x/A4k1EQ).

### Load Reference L1B Output

In [ ]:
output_file = Path.cwd().parent / "tests" / "test_data" / "l1b_integration_data" / "LIBERA_L1B_RAD-4CH_V0-4-4_20251120T175950_20251120T190549_R26019201635.nc"
output_data = xr.open_dataset(output_file)
print(f"Dataset loaded successfully")
print(f"\nDataset dimensions: {dict(output_data.sizes)}")

# Create DataFrame with radiance values
df_radiance = pd.DataFrame({
    'time': output_data['radiometer_time'],
    'radiance_ch0': output_data["Filtered_Radiance_SW"],
    'radiance_ch1': output_data["Filtered_Radiance_Tot"],
    'radiance_ch2': output_data["Filtered_Radiance_LW"],
    'radiance_ch3': output_data["Filtered_Radiance_SSW"]
}).set_index('time')

### Visualise Reference Radiance

Plot the radiance from the reference file using the same format as the notebook output above. Visual agreement in amplitude, shape, and noise level is the first check.

In [ ]:
# Plot first 10 seconds of radiance data
#n_samples_plot = min(1000, len(df_radiance))  # 1000 samples at 100 Hz = 10 seconds
n_samples_plot = len(df_radiance)
df_plot = df_radiance.iloc[:n_samples_plot]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Calibrated Radiometer Data - Radiance (100 Hz)', fontsize=14, fontweight='bold')

channels = ['ch0', 'ch1', 'ch2', 'ch3']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, (ax, ch, label, color) in enumerate(zip(axes, channels, channel_names, colors)):
    ax.plot(df_plot.index, df_plot[f'radiance_{ch}'],
            color=color, linewidth=0.8, alpha=0.8, label=label)
    ax.set_ylabel(f'{label}\n(W m^-2 sr^-1)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=9)

    mean_val = df_plot[f'radiance_{ch}'].mean()
    std_val = df_plot[f'radiance_{ch}'].std()
    ax.text(0.02, 0.95, f'μ={mean_val:.4f}, σ={std_val:.6f}',
            transform=ax.transAxes, fontsize=9,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[-1].set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
axes[-1].xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {n_samples_plot} samples at 100 Hz ({n_samples_plot/100:.1f} seconds)")


n_samples_plot = min(1000, len(df_radiance))
df_plot = df_radiance.iloc[:n_samples_plot]

fig, ax = plt.subplots(1, 1, figsize=(14, 6))
fig.suptitle('All Channels - Radiance Comparison (100 Hz)', fontsize=14, fontweight='bold')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
channels = ['ch0', 'ch1', 'ch2', 'ch3']
labels = ['SW', 'TOTAL', 'LW', 'SSW']

for ch, label, color in zip(channels, labels, colors):
    ax.plot(df_plot.index, df_plot[f'radiance_{ch}'],
            label=label, color=color, linewidth=1.0, alpha=0.8)

ax.set_ylabel('Radiance (W m^-2 sr^-1)', fontsize=11, fontweight='bold')
ax.set_xlabel('Time (UTC)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Plotted {n_samples_plot} samples at 100 Hz ({n_samples_plot/100:.1f} seconds)")


print("Statistical Comparison: Calibrated DN vs Radiance")
print("=" * 70)

for i, ch in enumerate(['sw', 'total', 'lw', 'ssw']):
    rad = radiance_100hz[ch]
    if i == 0:
        dn = rad_ch0_calibrated_100hz
    elif i == 1:
        dn = rad_ch1_calibrated_100hz
    elif i == 2:
        dn = rad_ch2_calibrated_100hz
    else:
        dn = rad_ch3_calibrated_100hz


    print(f"\{ch.upper()}:")
    print(f"  DN Statistics:")
    print(f"    Mean: {np.mean(dn):.3f} DN")
    print(f"    Std:  {np.std(dn):.4f} DN")
    print(f"    CV:   {(np.std(dn)/np.mean(dn)*100):.4f}%")
    print(f"  Radiance Statistics:")
    print(f"    Mean: {np.mean(rad):.6f} W m^-2 sr^-1")
    print(f"    Std:  {np.std(rad):.8f} W m^-2 sr^-1")
    print(f"    CV:   {(np.std(rad)/np.mean(rad)*100):.4f}%")
    print(f"  Signal-to-Noise Ratio: {np.mean(rad)/np.std(rad):.1f}")

### Full-Record Pipeline Comparison

Reset the time window to cover the full observation record (rather than the 2-second window used earlier) and re-run the per-channel pipeline plots against the reference radiance. This confirms end-to-end agreement across the entire file, not just a short excerpt.

In [ ]:
# Select time window for comparison (first 2 seconds = 400 samples at 200 Hz)
n_samples_200hz = len(df_radiance) * 2
n_samples_100hz = len(df_radiance)

# Get timestamps
time_200hz = radiometer_timestamps[:n_samples_200hz]
time_100hz = radiometer_timestamps_100hz[:n_samples_100hz]

In [ ]:
# Get data for each processing stage
ch0_raw_dn = rad_ch0[:n_samples_200hz]
ch0_calibrated_dn = rad_ch0_calibrated[:n_samples_200hz]
ch0_downsampled_dn = rad_ch0_calibrated_100hz[:n_samples_100hz]
ch0_radiance = df_radiance['radiance_ch0'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch0_raw_dn},
        {"time": time_200hz, "data": ch0_calibrated_dn},
        {"time": time_100hz, "data": ch0_downsampled_dn},
        {"time": time_100hz, "data": ch0_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Shortwave (Channel 0)",
)

print(f"Channel 0 (Shortwave) - Processing stages comparison:")
print(f"  Time window: {n_samples_200hz/200:.1f} seconds")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch0_raw_dn):.3f} to {np.std(ch0_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch0_radiance):.6f} ± {np.std(ch0_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch1_raw_dn = rad_ch1[:n_samples_200hz]
ch1_calibrated_dn = rad_ch1_calibrated[:n_samples_200hz]
ch1_downsampled_dn = rad_ch1_calibrated_100hz[:n_samples_100hz]
ch1_radiance = df_radiance['radiance_ch1'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch1_raw_dn},
        {"time": time_200hz, "data": ch1_calibrated_dn},
        {"time": time_100hz, "data": ch1_downsampled_dn},
        {"time": time_100hz, "data": ch1_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Total (Channel 1)",
)

print(f"Channel 1 (Total) - Processing stages comparison:")
print(f"  Time window: {n_samples_200hz/200:.1f} seconds")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch1_raw_dn):.3f} to {np.std(ch1_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch1_radiance):.6f} ± {np.std(ch1_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch2_raw_dn = rad_ch2[:n_samples_200hz]
ch2_calibrated_dn = rad_ch2_calibrated[:n_samples_200hz]
ch2_downsampled_dn = rad_ch2_calibrated_100hz[:n_samples_100hz]
ch2_radiance = df_radiance['radiance_ch2'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch2_raw_dn},
        {"time": time_200hz, "data": ch2_calibrated_dn},
        {"time": time_100hz, "data": ch2_downsampled_dn},
        {"time": time_100hz, "data": ch2_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Longwave (Channel 2)",
)

print(f"Channel 2 (Longwave) - Processing stages comparison:")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch2_raw_dn):.3f} to {np.std(ch2_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch2_radiance):.6f} ± {np.std(ch2_radiance):.8f} W m^-2 sr^-1")

# Get data for each processing stage
ch3_raw_dn = rad_ch3[:n_samples_200hz]
ch3_calibrated_dn = rad_ch3_calibrated[:n_samples_200hz]
ch3_downsampled_dn = rad_ch3_calibrated_100hz[:n_samples_100hz]
ch3_radiance = df_radiance['radiance_ch3'][:n_samples_100hz]

fig, axes = plot_processing_pipeline(
    stages=[
        {"time": time_200hz, "data": ch3_raw_dn},
        {"time": time_200hz, "data": ch3_calibrated_dn},
        {"time": time_100hz, "data": ch3_downsampled_dn},
        {"time": time_100hz, "data": ch3_radiance, "stat_fmt": ".6f"},
    ],
    channel_name="Split Shortwave (Channel 3)",
)

print(f"Channel 3 (Split Shortwave) - Processing stages comparison:")
print(f"  Raw DN → Calibrated DN: Noise reduced from {np.std(ch3_raw_dn):.3f} to {np.std(ch3_calibrated_dn):.3f} DN")
print(f"  Final radiance: {np.mean(ch3_radiance):.6f} ± {np.std(ch3_radiance):.8f} W m^-2 sr^-1")

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex='col')
fig.suptitle('Complete L1B Processing Pipeline - All Channels', fontsize=18, fontweight='bold')

# Define channels
channels_data = [
    {
        'name': 'SW',
        'raw': rad_ch0[:n_samples_200hz],
        'calibrated': rad_ch0_calibrated[:n_samples_200hz],
        'downsampled': rad_ch0_calibrated_100hz[:n_samples_100hz],
        'radiance': df_radiance['radiance_ch0'][:n_samples_100hz],
        'color': '#1f77b4'
    },
    {
        'name': 'TOTAL',
        'raw': rad_ch1[:n_samples_200hz],
        'calibrated': rad_ch1_calibrated[:n_samples_200hz],
        'downsampled': rad_ch1_calibrated_100hz[:n_samples_100hz],
        'radiance': df_radiance['radiance_ch1'][:n_samples_100hz],
        'color': '#ff7f0e'
    },
    {
        'name': 'LW',
        'raw': rad_ch2[:n_samples_200hz],
        'calibrated': rad_ch2_calibrated[:n_samples_200hz],
        'downsampled': rad_ch2_calibrated_100hz[:n_samples_100hz],
        'radiance': df_radiance['radiance_ch2'][:n_samples_100hz],
        'color': '#2ca02c'
    },
    {
        'name': 'SSW',
        'raw': rad_ch3[:n_samples_200hz],
        'calibrated': rad_ch3_calibrated[:n_samples_200hz],
        'downsampled': rad_ch3_calibrated_100hz[:n_samples_100hz],
        'radiance': df_radiance['radiance_ch3'][:n_samples_100hz],
        'color': '#d62728'
    }
]

stage_names = ['Raw DN\n(200 Hz)', 'Calibrated DN\n(200 Hz)',
               'Downsampled DN\n(100 Hz)', 'Radiance\n(W m$^-2$ sr$^-1$)\n(100 Hz)']
stage_keys = ['raw', 'calibrated', 'downsampled', 'radiance']

# Plot each stage for each channel
for stage_idx, (stage_name, stage_key) in enumerate(zip(stage_names, stage_keys)):
    for ch_idx, ch_data in enumerate(channels_data):
        ax = axes[stage_idx, ch_idx]

        # Select appropriate time array
        if stage_key in ['raw', 'calibrated']:
            time_array = time_200hz
        else:
            time_array = time_100hz

        # Plot data
        data = ch_data[stage_key]
        ax.plot(time_array, data, color=ch_data['color'],
                linewidth=0.8 if stage_key in ['raw', 'calibrated'] else 1.0,
                alpha=0.8)

        # Set title for top row
        if stage_idx == 0:
            ax.set_title(ch_data['name'], fontsize=12, fontweight='bold')

        # Set ylabel for leftmost column
        if ch_idx == 0:
            ax.set_ylabel(stage_name, fontsize=10, fontweight='bold')

        # Add grid
        ax.grid(True, alpha=0.3)

        # Add statistics
        mean_val = np.mean(data)
        std_val = np.std(data)
        if stage_key == 'radiance':
            label = f'{mean_val:.4f}\n±{std_val:.6f}'
        else:
            label = f'{mean_val:.1f}\n±{std_val:.2f}'

        ax.text(0.98, 0.95, label, transform=ax.transAxes,
                fontsize=8, verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

        # Format x-axis for bottom row
        if stage_idx == 3:
            ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
            ax.set_xlabel('Time (UTC)', fontsize=9)

plt.tight_layout()
plt.show()

print("\nProcessing Pipeline Summary:")
print("=" * 80)
for ch_idx, ch_data in enumerate(channels_data):
    print(f"\n{ch_data['name']}:")
    print(f"  Raw DN:                      {np.mean(ch_data['raw']):.2f} ± {np.std(ch_data['raw']):.3f} DN")
    print(f"  Raw DN Data Points:          {len(ch_data['raw'])}")
    print(f"  Calibrated DN:               {np.mean(ch_data['calibrated']):.2f} ± {np.std(ch_data['calibrated']):.3f} DN")
    print(f"  Calibrated DN Data Points:   {len(ch_data['calibrated'])}")
    print(f"  Downsampled DN:              {np.mean(ch_data['downsampled']):.2f} ± {np.std(ch_data['downsampled']):.3f} DN")
    print(f"  Downsampled DN Data Points:  {len(ch_data['downsampled'])}")
    print(f"  Radiance:                    {np.mean(ch_data['radiance']):.6f} ± {np.std(ch_data['radiance']):.8f} W m⁻² sr⁻¹")
    print(f"  Radiance Data Points:        {len(ch_data['radiance'])}")
    print(f"  Noise reduction:             {np.std(ch_data['raw'])/np.std(ch_data['calibrated']):.2f}x")


---
## Tier 1 Validation — Sampling Frequency Verification

This section verifies that the data are correctly sampled at the expected rates (200 Hz before downsampling, 100 Hz after). It is a requirement for the Tier 1 test that the downsampling occurs and is correct in the output L1B file.

### Helper: `calculate_sampling_frequency()`

Computes instantaneous inter-sample frequencies from a timestamp array and returns summary statistics (mean, std, median, min, max) as well as the full frequency array for histogram plotting.

In [ ]:
def calculate_sampling_frequency(time_array):
    """
    Calculate sampling frequency from timestamp array.

    Parameters
    ----------
    time_array : pd.DatetimeIndex or array of timestamps
        Array of timestamps

    Returns
    -------
    dict
        Dictionary containing:
        - 'mean_hz': Mean sampling frequency in Hz
        - 'std_hz': Standard deviation of sampling frequency
        - 'median_hz': Median sampling frequency
        - 'min_hz': Minimum sampling frequency
        - 'max_hz': Maximum sampling frequency
        - 'freq_array': Array of instantaneous frequencies
    """
    # Calculate time differences in seconds
    time_diffs = np.diff(time_array).astype('timedelta64[ns]').astype(float) / 1e9

    # Calculate instantaneous frequencies (1 / time_diff)
    freq_array = 1.0 / time_diffs

    return {
        'mean_hz': np.mean(freq_array),
        'std_hz': np.std(freq_array),
        'median_hz': np.median(freq_array),
        'min_hz': np.min(freq_array),
        'max_hz': np.max(freq_array),
        'freq_array': freq_array
    }

### Sampling Frequency Statistics and Histogram

The statistics table confirms the measured mean sampling frequencies are within tolerance of the 200 Hz and 100 Hz nominals. The 4×4 histogram grid plots the distribution of inter-sample frequencies for each channel at each processing stage. A single value at nominal frequency (red dashed line) indicates correct downsampling. Significant spread or offset would suggest a timing issue in the upstream decode pipeline.

In [ ]:

# Calculate sampling frequencies for each processing stage

# Calculate frequencies for each stage
freq_200hz = calculate_sampling_frequency(radiometer_timestamps)
freq_100hz = calculate_sampling_frequency(radiometer_timestamps_100hz)

print("Sampling Frequency Statistics:")
print("=" * 80)
print(f"\nRaw DN & Calibrated DN (200 Hz nominal):")
print(f"  Mean:   {freq_200hz['mean_hz']:.4f} Hz")
print(f"  Std:    {freq_200hz['std_hz']:.6f} Hz")
print(f"  Median: {freq_200hz['median_hz']:.4f} Hz")
print(f"  Range:  [{freq_200hz['min_hz']:.4f}, {freq_200hz['max_hz']:.4f}] Hz")

print(f"\nDownsampled DN & Radiance (100 Hz nominal):")
print(f"  Mean:   {freq_100hz['mean_hz']:.4f} Hz")
print(f"  Std:    {freq_100hz['std_hz']:.6f} Hz")
print(f"  Median: {freq_100hz['median_hz']:.4f} Hz")
print(f"  Range:  [{freq_100hz['min_hz']:.4f}, {freq_100hz['max_hz']:.4f}] Hz")

# Create 4×4 grid plot showing sampling frequency for each channel/stage

fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex='row')
fig.suptitle('L1B Radiometer Sampling Frequency - All Channels & Processing Stages',
             fontsize=18, fontweight='bold')

# Channel info (same structure as before)
channels_data = [
    {'name': 'SW',     'color': '#1f77b4'},
    {'name': 'TOTAL',  'color': '#ff7f0e'},
    {'name': 'LW',     'color': '#2ca02c'},
    {'name': 'SSW',    'color': '#d62728'},
]

stage_names = [
    'Raw DN\n(200 Hz nominal)',
    'Calibrated DN\n(200 Hz nominal)',
    'Downsampled DN\n(100 Hz nominal)',
    'Radiance\n(100 Hz nominal)'
]

# Plot each stage (all channels share same freq for a given stage)
for stage_idx, stage_name in enumerate(stage_names):
    # Select appropriate frequency data
    if stage_idx in [0, 1]:  # Raw DN and Calibrated DN at 200 Hz
        freq_data = freq_200hz
        expected_freq = 200.0
    else:  # Downsampled DN and Radiance at 100 Hz
        freq_data = freq_100hz
        expected_freq = 100.0

    for ch_idx, ch_data in enumerate(channels_data):
        ax = axes[stage_idx, ch_idx]

        # Create histogram of sampling frequencies
        counts, bins, patches = ax.hist(
            freq_data['freq_array'],
            bins=50,
            color=ch_data['color'],
            alpha=0.7,
            edgecolor='black',
            linewidth=0.5,
            range=(50, 250)
        )

        # Add vertical line at expected frequency
        ax.axvline(expected_freq, color='red', linestyle='--',
                   linewidth=2, alpha=0.8, label=f'Nominal {expected_freq} Hz')

        # Add vertical line at mean frequency
        ax.axvline(freq_data['mean_hz'], color='darkgreen', linestyle='-',
                   linewidth=2, alpha=0.8, label=f"Mean {freq_data['mean_hz']:.2f} Hz")

        # Set title for top row
        if stage_idx == 0:
            ax.set_title(ch_data['name'], fontsize=12, fontweight='bold')

        # Set ylabel for leftmost column
        if ch_idx == 0:
            ax.set_ylabel(f"{stage_name}\nCount", fontsize=10, fontweight='bold')
        else:
            ax.set_ylabel('Count', fontsize=9)

        # Add grid
        ax.grid(True, alpha=0.3, axis='y')

        # Format x-axis for bottom row
        if stage_idx == 3:
            ax.set_xlabel('Frequency (Hz)', fontsize=9)

        # Add legend only to first column
        if ch_idx == 0:
            ax.legend(fontsize=7, loc='upper left', framealpha=0.9)

plt.tight_layout()
plt.show()

# Print detailed comparison
print("\n" + "=" * 80)
print("Sampling Frequency Stability Analysis:")
print("=" * 80)

for stage_name, freq_data, nominal_freq in [
    ('Raw DN (200 Hz)',         freq_200hz, 200.0),
    ('Calibrated DN (200 Hz)',  freq_200hz, 200.0),
    ('Downsampled DN (100 Hz)', freq_100hz, 100.0),
    ('Radiance (100 Hz)',       freq_100hz, 100.0),
]:
    deviation = freq_data['mean_hz'] - nominal_freq
    deviation_pct = (deviation / nominal_freq) * 100
    stability = (freq_data['std_hz'] / freq_data['mean_hz']) * 100  # Coefficient of variation

    print(f"\n{stage_name}:")
    print(f"  Nominal frequency:     {nominal_freq:.2f} Hz")
    print(f"  Measured mean:         {freq_data['mean_hz']:.6f} Hz")
    print(f"  Deviation from nominal: {deviation:+.6f} Hz ({deviation_pct:+.4f}%)")
    print(f"  Std deviation:         {freq_data['std_hz']:.6f} Hz")
    print(f"  Stability (CV):        {stability:.6f}%")
    print(f"  Range:                 [{freq_data['min_hz']:.4f}, {freq_data['max_hz']:.4f}] Hz")
